# 01 – Train NLP Model for BANKING77 Intent Classification

This notebook trains a **TF-IDF + Logistic Regression** classifier on the BANKING77 dataset.

- 10,003 training examples
- 3,080 test examples
- 77 fine-grained intents

The trained pipeline is saved to `models/support_ticket_classifier/` for use by the Flask app.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import LabelEncoder
import joblib

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / "support_ticket_classifier"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 1. Load Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {len(train_df)} examples")
print(f"Test:  {len(test_df)} examples")
print(f"Unique intents: {train_df['category'].nunique()}")
train_df.head()

## 2. Encode Labels

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(train_df["category"])
y_test = le.transform(test_df["category"])
labels = list(le.classes_)
print(f"Number of classes: {len(labels)}")
print("Sample labels:", labels[:8])

## 3. Train Pipeline

In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=15000,
        ngram_range=(1, 2),
        stop_words="english",
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        C=2.0,
        solver="saga",
    )),
])

print("Training...")
pipe.fit(train_df["text"], y_train)
print("Done.")

## 4. Evaluate

In [ ]:
y_pred = pipe.predict(test_df["text"])
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy : {acc:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")

report = classification_report(
    y_test, y_pred, target_names=labels, zero_division=0
)
print(report)

## 5. Confusion Matrix (sample of classes)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix shape:", cm.shape)
# Full 77x77 matrix is large; look at diagonal dominance
print("Mean diagonal (correct predictions per class):", np.diag(cm).mean())

## 6. Save Model & Artifacts

In [ ]:
joblib.dump(pipe, MODEL_DIR / "pipeline.joblib")
joblib.dump(le, MODEL_DIR / "label_encoder.joblib")

with open(MODEL_DIR / "labels.json", "w") as f:
    json.dump(labels, f, indent=2)

metrics = {
    "accuracy": float(acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(f1_score(y_test, y_pred, average="weighted")),
    "precision_macro": float(
        classification_report(y_test, y_pred, output_dict=True, zero_division=0)["macro avg"]["precision"]
    ),
    "recall_macro": float(
        classification_report(y_test, y_pred, output_dict=True, zero_division=0)["macro avg"]["recall"]
    ),
    "num_classes": len(labels),
    "train_size": len(train_df),
    "test_size": len(test_df),
    "model_type": "TF-IDF + LogisticRegression (saga)",
}

with open(MODEL_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved to", MODEL_DIR)
print(json.dumps(metrics, indent=2))

## 7. Quick Inference Test

In [ ]:
def predict_top_k(text, k=3):
    proba = pipe.predict_proba([text])[0]
    top_idx = np.argsort(proba)[::-1][:k]
    return [(labels[i], float(proba[i])) for i in top_idx]

samples = [
    "My card hasn't arrived yet",
    "I was charged an unexpected fee",
    "How do I change my PIN?",
    "The transfer still has not arrived",
]

for s in samples:
    print(f"\nQuery: {s}")
    for intent, conf in predict_top_k(s):
        print(f"  {intent:45s} {conf*100:5.1f}%")